# PLN Sesión 3: Traducción Automática (Evaluación y Post-edición)

---

## ¿Por qué?

La traducción automática ha pasado de ser una curiosidad tecnológica a una herramienta central en la industria de la lengua. Sin embargo, el paso de los sistemas basados en reglas a los sistemas neuronales (NMT) ha creado un fenómeno nuevo: la **trampa de la fluidez**. 

Un sistema moderno puede producir una frase gramaticalmente impecable y elegante que, sin embargo, omite una negación crucial o altera el sentido original. El profesional de la lengua ya no es solo un traductor, sino un **árbitro** que audita la fidelidad y la calidad del output de la máquina.

## ¿Qué?

En esta sesión se exploran dos pilares de la traducción profesional contemporánea:
1. **Métricas de calidad (BLEU):** Cómo los algoritmos intentan puntuar una traducción comparándola con una referencia humana.
2. **Post-edición (PE):** El proceso de corrección humana de textos traducidos automáticamente para alcanzar un estándar profesional.

## ¿Para qué?

Comprender estos conceptos permite:
- **Evaluar motores de traducción:** Saber por qué un sistema puntúa alto aunque el sentido falle.
- **Sistematizar la corrección:** Identificar y categorizar tipos de error (léxicos, sintácticos, de registro) en lugar de corregir por intuición.
- **Optimizar flujos de trabajo:** Decidir cuándo es rentable usar traducción automática y cuándo el coste de post-edición supera al de la traducción humana.

## ¿Cómo?

Se analizará el funcionamiento de la métrica BLEU mediante un calculador simplificado y se realizará un ejercicio de post-edición con etiquetado de errores sobre casos reales.

---
## 1. Preparación del entorno

Para este análisis se requieren herramientas de procesamiento de texto básicas y la capacidad de comparar n-gramas (secuencias de palabras).

In [ ]:
import pandas as pd
from collections import Counter

def obtener_ngramas(texto, n):
    """Divide un texto en secuencias de n palabras."""
    palabras = texto.lower().replace(".", "").replace(",", "").split()
    return [tuple(palabras[i:i+n]) for i in range(len(palabras)-n+1)]

print("Funciones de análisis preparadas.")

---
## 2. El principio detrás de BLEU: Geometría de la coincidencia

La métrica **BLEU** (*Bilingual Evaluation Understudy*) es el estándar industrial para medir la calidad de la traducción automática. Su lógica es puramente estadística: cuenta cuántas secuencias de palabras (n-gramas) de la máquina coinciden con las de una traducción humana de referencia.

### Limitación estructural
BLEU no entiende el sentido; solo mide coincidencia léxica. Si la máquina usa un sinónimo perfecto que no está en la referencia, la puntuación baja. Si la máquina mantiene las palabras pero altera el orden o la polaridad (añade o quita un "no"), la puntuación puede seguir siendo engañosamente alta.

Considérese el siguiente experimento donde comparamos desde unigramas (palabras sueltas) hasta trigramas (grupos de tres palabras):

In [ ]:
referencia_humana = "El sistema de inteligencia artificial detectó un error crítico."
traduccion_maquina_1 = "El sistema de inteligencia artificial detectó un error crítico." # Perfecta
traduccion_maquina_2 = "La plataforma de IA identificó un fallo grave."         # Sinónimos (Sentido OK)
traduccion_maquina_3 = "El sistema de inteligencia artificial no detectó un error." # Omisión (Sentido MAL)

def calcular_overlap(candidato, referencia, n):
    ng_cand = obtener_ngramas(candidato, n)
    ng_ref = obtener_ngramas(referencia, n)
    
    if not ng_ref:
        return 0.0
        
    coincidencias = set(ng_cand) & set(ng_ref)
    return len(coincidencias) / len(ng_ref)

ejemplos = [traduccion_maquina_1, traduccion_maquina_2, traduccion_maquina_3]
resultados = []

for i, m in enumerate(ejemplos, 1):
    resultados.append({
        "Traducción": m,
        "Unigramas (%)": f"{calcular_overlap(m, referencia_humana, 1)*100:.1f}",
        "Bigramas (%)": f"{calcular_overlap(m, referencia_humana, 2)*100:.1f}",
        "Trigramas (%)": f"{calcular_overlap(m, referencia_humana, 3)*100:.1f}"
    })

pd.DataFrame(resultados)

### Reflexión técnica
- La **traducción 2** tiene un sentido excelente, pero su puntuación es baja porque el modelo estadístico no reconoce sinónimos.
- La **traducción 3** tiene un sentido opuesto (omite el error crítico). Obsérvese cómo su puntuación experimenta una **caída progresiva**: es alta en unigramas (88%), disminuye en bigramas (75%) y se sitúa en el **57% en trigramas**. La coincidencia secuencial se debilita a medida que aumenta la longitud del fragmento analizado.
- **Conclusión:** La máquina puntúa coincidencia; el profesional evalúa sentido.

---
## 3. Reto: Auditoría de Post-edición

La labor profesional consiste en la **Post-edición**: detectar errores en el output automático, corregirlos y categorizarlos para mejorar el motor de traducción.

### Ejercicio:
1. Analícese el siguiente fragmento traducido por un motor neuronal.
2. Identifíquense los errores (hay al menos un error crítico de sentido y uno de registro).
3. Realícese la post-edición y clasifíquese el error principal.

In [ ]:
texto_origen = """
The patient was discharged after the surgery, but the clinical staff noticed a slight 
swelling in the lower limb. No further intervention was required at that stage.
"""

traduccion_automatica = """
El paciente fue disparado después de la cirugía, pero el personal clínico notó una 
ligera hinchazón en el miembro inferior. No se requirió más intervención en esa etapa.
"""

print("--- Traducción a evaluar ---")
print(traduccion_automatica)

post_edicion_humana = """
ESCRIBE AQUÍ TU VERSIÓN CORREGIDA
"""

if "disparado" in post_edicion_humana.lower():
    print("Alerta: Un error crítico de sentido aún persiste en el texto.")
elif "miembro inferior" in post_edicion_humana.lower():
    print("Nota: El texto es correcto, pero ¿se ha ajustado el registro al estándar clínico 'extremidad inferior'?")
elif post_edicion_humana.strip() == "ESCRIBE AQUÍ TU VERSIÓN CORREGIDA":
    print("Por favor, completa la post-edición.")
else:
    print("Post-edición completada.")

### Clasificación del error principal
Defínase el tipo de error predominante en la variable `tipo_error` del siguiente bloque. 

Opciones: `"lexico"` (falso amigo), `"gramatical"`, `"registro"` o `"terminologico"`.

In [ ]:
tipo_error = "" # Escribe aquí la categoría

if tipo_error == "lexico":
    print("Correcto: 'discharged' -> 'disparado' es un error léxico crítico (falso amigo).")
elif tipo_error in ["registro", "terminologico"]:
    print("Identificado: El uso de 'miembro' vs 'extremidad' es un matiz de registro o terminología.")
else:
    print("Indica una categoría para validar el análisis.")

---
## 4. Reto Final: Auditoría sobre datos propios

Para cerrar el bloque, aplíquese el mismo protocolo sobre un ejemplo real de interés personal.

### Procedimiento:
1. Búsquese un fragmento breve en inglés (o cualquier otra lengua) de un dominio conocido (literario, jurídico, técnico, periodístico).
2. Tradúzcase mediante un sistema comercial (Google Translate, DeepL, ChatGPT).
3. Péguese el resultado y realícese la auditoría profesional.

In [ ]:
texto_original_usuario = """ PEGA AQUÍ EL ORIGINAL """
traduccion_maquina_usuario = """ PEGA AQUÍ LA TRADUCCIÓN AUTOMÁTICA """
post_edicion_usuario = """ ESCRIBE AQUÍ TU CORRECCIÓN """

# Clasificación de errores encontrados
errores_detectados = [
    "Error 1: descripción",
    "Error 2: descripción"
]

plantilla = ["PEGA AQUÍ", "ESCRIBE AQUÍ", "Error 1: descripción"]
if any(p in texto_original_usuario or p in post_edicion_usuario or p in "".join(errores_detectados) for p in plantilla):
    print("Por favor, introduce tus propios datos y completa la lista de errores para el reto final.")
else:
    print("Auditoría de usuario registrada con éxito.")

---
## Conclusiones del Bloque de PLN

### Nota técnica: De la estructura a la generación

A lo largo de estas tres sesiones se ha recorrido un camino de abstracción creciente:
1. **Estructura (Sesión 1):** Identificación de entidades y funciones gramaticales.
2. **Concepto (Sesión 2):** Agrupación por significado mediante vectores (embeddings).
3. **Producción (Sesión 3):** Evaluación crítica de la traducción y juicio sobre la calidad del output.

Este dominio sobre la **Ingeniería Lingüística** es la base necesaria para comprender la **IA Generativa**. Los Grandes Modelos de Lenguaje (LLMs) que se estudiarán en el próximo bloque no son más que sistemas que combinan estos tres niveles a una escala masiva. 

El profesional de la lengua del siglo XXI no compite con la máquina; la diseña, la audita y la perfecciona.